# Songs 1970–1975: clusters by lyrics, instruments and country

**Research question.** To what extent can songs from 1970–1975 be grouped into meaningful clusters based on lyrical themes, country of origin and instrumentation, and how do these clusters correspond to established musical genres?

1. Clustering: do the song graphs have genuine community structure?
2. Associations between instrument, theme and country.
3. Alignment of the clusters with genre.

This notebook runs the whole analysis in order; the functions live in `src/`. The data decisions (genre labels, country, instrument families, which songs enter which graph) are explained in `Documentation.ipynb`. All randomness is driven by `SEED`. Figures go to `figures/` (PDF + PNG), tables to `results/`.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import networkx as nx

from src.config import SEED, FIG_DIR, RESULTS_DIR
from src.dataset import load_analysis_songs
from src import graphs

FIG_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)
pd.set_option("display.width", 160)
print("SEED =", SEED)

SEED = 42


## Data

One row per song first released 1970–1975 (MusicBrainz), with Genius lyrics where matched. Songs whose lyrics are a near-copy of another song by the same artist are dropped as duplicates (spelling variants the title matching missed). Covers by other artists are kept.

Filler words are removed from the tokens when `config.REMOVE_FILLER_WORDS` is True. These are vocables like "ooh", "la" and "yeah", plus what spaCy leaves of contractions, like "gon" + "na" from "gonna" (lists in `src/text.py`). They make up about 10% of the tokens, and they would otherwise be among the most common top-10 words. Set the flag to False to undo it.

`genre_eval` is the evaluation genre: the vote-based coarse genre, with classes under 30 songs merged into "other". **Genre is only metadata: it is never used to build an edge.**

In [2]:
songs = load_analysis_songs()
lyr = songs[songs.has_lyrics]

coverage = pd.Series({
    "songs": len(songs),
    "with lyrics": songs.has_lyrics.sum(),
    "with instruments": songs.has_instruments.sum(),
    "with lyrics and instruments": (songs.has_lyrics & songs.has_instruments).sum(),
}, name="n")
display(coverage.to_frame())
display(lyr.genre_eval.value_counts(dropna=False).rename("songs with lyrics").to_frame())

Filler words removed: 10.5% of the lyrics tokens (config.REMOVE_FILLER_WORDS)


11032 songs 1970-1975 (7 lyric duplicates dropped): 3227 with lyrics, 735 with instruments, 519 with both


,n
songs,11032
with lyrics,3227
with instruments,735
with lyrics and instruments,519


,songs with lyrics
genre_eval,
rock,1567
rb,571
country,331
pop,325
NaN,234
folk,85
other,73
reggae,41


## Step 1 — Three song graphs

| Graph | Songs | Edges |
|---|---|---|
| `G_lyrics` | all songs with lyrics | cosine similarity of TF-IDF vectors |
| `G_instruments` | all songs with instrument data | hyperbolic projection of song–instrument family |
| `G_combined` | songs with lyrics, instruments and country | hyperbolic projection of song–(instrument family, country, top-10 words) |

There is no country-only graph: projecting song–country would just give one clique per country. Country is used as metadata (like genre) and as a feature inside `G_combined`.

### G_lyrics

TF-IDF on the spaCy tokens (used as they are), with `min_df=2` (a word must occur in two songs to link them), `max_df=0.5` (words in more than half the songs are near-stop words) and `sublinear_tf=True` (tf = 1 + ln count, so a repeated chorus word cannot dominate). Rows are L2-normalised, so their dot product is the cosine similarity.

Almost every pair of songs shares some word: the full similarity matrix is 91% non-zero (≈4.8M edges). That is too dense for networkx and for the null models, so each song keeps its 20 most similar songs as candidate edges (kept if either song chose it). The disparity filter in Step 2 then decides which of these edges are significant.

In [3]:
X, terms = graphs.tfidf(lyr.tokens)
G_lyrics = graphs.add_metadata(graphs.lyrics_graph(X, lyr.index.to_numpy(), k=graphs.K_NEIGHBOURS), songs)
print(f"TF-IDF vocabulary: {len(terms)} words")
graphs.summary(G_lyrics)

TF-IDF vocabulary: 6942 words


{'nodes': 3227,
 'edges': 48994,
 'density': 0.009412599173395738,
 'components': 2,
 'isolated': 1,
 'lcc_nodes': 3226,
 'lcc_share': 0.9996901146575767}

### G_instruments

Bipartite graph of songs and instrument families (`bipartite=0/1`), projected onto the songs with Newman's hyperbolic weighting (`collaboration_weighted_projected_graph`). A family shared by *k* songs adds 1/(*k*−1) to each of their pairs. So guitar, bass and drums, which are on 80–85% of the songs, add very little to any single pair, while a rare shared family (harmonica, folk strings) adds a lot. The projection is still almost complete, because nearly every pair shares guitar, bass or drums. The disparity filter will remove those weak links.

In [4]:
B_instruments = graphs.bipartite_graph(graphs.instrument_features(songs))
G_instruments = graphs.add_metadata(graphs.project(B_instruments), songs)
graphs.summary(G_instruments)

{'nodes': 735,
 'edges': 247574,
 'density': 0.9178075589908988,
 'components': 1,
 'isolated': 0,
 'lcc_nodes': 735,
 'lcc_share': 1.0}

### G_combined

Songs link to three kinds of feature nodes: `inst:<family>`, `country:<release country>` and `word:<w>` for the song's 10 highest-TF-IDF words. The words are the lyrical themes, taken from the same TF-IDF as `G_lyrics`, so the IDF is estimated on all songs with lyrics. The same hyperbolic projection is used.

A feature of degree *k* gives its *k*(*k*−1)/2 pairs weight 1/(*k*−1) each, i.e. *k*/2 in total. Common features therefore add little to each pair but a lot to the total. The table below shows each feature type's share of the total edge weight before backboning.

In [5]:
words = dict(zip(lyr.index, graphs.top_words(X, terms)))
B_combined = graphs.bipartite_graph(graphs.combined_features(songs, words))
G_combined = graphs.add_metadata(graphs.project(B_combined), songs)
display(pd.Series(graphs.feature_weight_shares(B_combined), name="share of edge weight").to_frame().round(3))
feature_nodes = [n for n, b in B_combined.nodes(data="bipartite") if b == 1]
print(f"{len(feature_nodes)} feature nodes, {sum(B_combined.degree(f) == 1 for f in feature_nodes)} "
      "of them on a single song (they link nobody)")
graphs.summary(G_combined)

,share of edge weight
word,0.556
inst,0.360
country,0.084


2867 feature nodes, 1808 of them on a single song (they link nobody)


{'nodes': 519,
 'edges': 126286,
 'density': 0.9394811822557487,
 'components': 1,
 'isolated': 0,
 'lcc_nodes': 519,
 'lcc_share': 1.0}

In [6]:
from collections import Counter
print("Most common top-10 words over the songs with lyrics:")
print(Counter(w for ws in words.values() for w in ws).most_common(25))

Most common top-10 words over the songs with lyrics:
[('baby', 203), ('love', 137), ('wanna', 97), ('girl', 81), ('woman', 79), ('want', 78), ('let', 77), ('need', 76), ('come', 72), ('yes', 70), ('life', 70), ('tell', 68), ('rock', 67), ('lord', 63), ('sweet', 63), ('little', 62), ('going', 62), ('home', 61), ('feel', 61), ('man', 60), ('sing', 60), ('mama', 59), ('everybody', 59), ('roll', 59), ('good', 59)]


In [7]:
step1 = pd.DataFrame({name: graphs.summary(G) for name, G in
                      [("G_lyrics", G_lyrics), ("G_instruments", G_instruments), ("G_combined", G_combined)]}).T
step1["avg_degree"] = 2 * step1.edges / step1.nodes
step1.to_csv(RESULTS_DIR / "step1_graphs.csv")
step1.round(4)

,nodes,edges,density,components,isolated,lcc_nodes,lcc_share,avg_degree
G_lyrics,3227.0,48994.0,0.0094,2.0,1.0,3226.0,0.9997,30.3650
G_instruments,735.0,247574.0,0.9178,1.0,0.0,735.0,1.0000,673.6707
G_combined,519.0,126286.0,0.9395,1.0,0.0,519.0,1.0000,486.6513
